# 05 — Train your own QSAR and run inverse design

You have an SDF with activity labels. Goal: encode → train sklearn → plug
the resulting model into the GA pipeline from tutorial 02. No VQGAE
retraining required — the published checkpoints already give you a useful
4096-d fragment-count representation.

**Data used here**: the 51-molecule `data/tubulin/colchicine_scaffold.sdf`
that ships with the repo. We synthesise binary labels from heavy-atom
count just to make the workflow runnable; for real work, plug in your
own SDF + your own labels at the marked spot.

For the narrative behind each step see [`docs/your_own_qsar.md`](../docs/your_own_qsar.md).


## 1. Imports + load the pretrained encoder

In [ ]:
import os
import pickle
import warnings
import numpy as np
from huggingface_hub import hf_hub_download
from chython.files import SDFRead
from VQGAE import VQGAE, vqgae_encode_dataset, frag_inds_to_counts

warnings.filterwarnings("ignore", category=FutureWarning)

REPO = "tagirshin/VQGAE"
vqgae_enc = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"),
    task="encode", batch_size=10, map_location="cpu",
).eval()

## 2. Encode your SDF

`vqgae_encode_dataset` takes any SDF and returns codebook indices `(N, 51)`.
`frag_inds_to_counts` turns those into the 4096-dim integer histogram that
your QSAR model trains on.


In [ ]:
# >>> EDIT THIS PATH for your real data <<<
MY_SDF = "../data/tubulin/colchicine_scaffold.sdf"

codebook_inds = vqgae_encode_dataset(MY_SDF, vqgae_enc, batch_size=10)
X = frag_inds_to_counts(codebook_inds).astype(np.int64)
print("X:", X.shape)

## 3. Read your labels

Real version: pull a numeric `mol.meta` field from your SDF. We
synthesise a label here so the demo runs anywhere.


In [ ]:
heavy_atom_counts = []
with SDFRead(MY_SDF, indexable=True) as inp:
    for mol in inp:
        heavy_atom_counts.append(len(mol))
heavy_atom_counts = np.array(heavy_atom_counts)

# >>> REAL VERSION <<<
# Y = []
# with SDFRead(MY_SDF, indexable=True) as inp:
#     for mol in inp:
#         Y.append(int(mol.meta["activity"]))    # adjust the key + cast
# Y = np.array(Y)

# Synthetic label for the demo: above-median heavy-atom count = 1
Y = (heavy_atom_counts > np.median(heavy_atom_counts)).astype(np.int64)
print(f"X: {X.shape}  Y: {Y.shape}  active fraction: {Y.mean():.2f}")
assert len(Y) == X.shape[0]

## 4. Train + persist the sklearn classifier

For a real dataset, drop in your own `RandomForestClassifier` /
`RandomForestRegressor` / `XGBClassifier` / etc. — anything with a
`predict_proba` (classification) or `predict` (regression) interface.

We keep the grid tiny here so the cell runs in a few seconds.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GridSearchCV

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = {"n_estimators": [50, 100], "max_features": ["sqrt"]}
rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    grid,
    scoring="balanced_accuracy",
    n_jobs=-1,
    cv=cv,
)
rf.fit(X, Y)
print(f"best params: {rf.best_params_}")
print(f"5-fold CV balanced accuracy: {rf.best_score_:.3f}")

OUT_PICKLE = "my_qsar_rf.pickle"
with open(OUT_PICKLE, "wb") as fh:
    pickle.dump(rf, fh)
print("wrote", OUT_PICKLE)

## 5. Plug into the GA pipeline

This is tutorial 02's fitness function with `rf_model` and `X` swapped for
yours. Drop into PyGAD, run a few generations, decode the survivors.


In [ ]:
import pygad
from VQGAE import (
    OrderingNetwork, decode_population,
    tanimoto_kernel, filter_molecule,
)

# Load decode side
vqgae_dec = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"),
    task="decode", batch_size=50, map_location="cpu",
).eval()
ordering_model = OrderingNetwork.load_from_checkpoint(
    hf_hub_download(REPO, "ordering_network.ckpt"),
    batch_size=50, map_location="cpu",
).eval()

with open(OUT_PICKLE, "rb") as fh:
    rf_model = pickle.load(fh)


def fitness_func_batch(_ga, solutions, _idx):
    counts = np.array(solutions)
    rf_score = rf_model.predict_proba(counts)[:, 1]
    size_pen = np.where(counts.sum(-1) < 18, -1.0, 0.0)
    dissim = 1 - tanimoto_kernel(counts, X).max(-1)
    dissim += np.where(dissim == 0, -5, 0)
    return (0.5 * rf_score + 0.3 * dissim + size_pen).tolist()


NUM_GENS = 3   # bump for real runs
ga = pygad.GA(
    fitness_func=fitness_func_batch,
    initial_population=X,
    num_genes=X.shape[-1],
    fitness_batch_size=50,
    num_generations=NUM_GENS,
    num_parents_mating=max(2, X.shape[0] // 3),
    parent_selection_type="rws",
    crossover_type="single_point",
    mutation_type="adaptive",
    mutation_percent_genes=[10, 5],
    save_solutions=True,
    keep_elitism=0,
    keep_parents=max(2, X.shape[0] // 5),
    suppress_warnings=True,
    random_seed=42,
    gene_type=int,
)
ga.run()
solutions = list({tuple(s) for s in ga.solutions})
print(f"GA: {len(solutions)} unique candidates")

## 6. Decode survivors and apply structure filters

In [ ]:
counts_arr = np.array(solutions, dtype=np.int64)
rf_scores  = rf_model.predict_proba(counts_arr)[:, 1]
sim_scores = tanimoto_kernel(counts_arr, X).max(-1)

mask = (rf_scores > 0.5) & (sim_scores < 0.95)
chosen = counts_arr[mask]
print(f"{len(chosen)} candidates pass coarse filters")

mols, validity, _ = decode_population(
    chosen, vqgae_dec, ordering_model, batch_size=50
) if len(chosen) > 0 else ([], [], [])

survivors = [m for m, ok in zip(mols, validity) if ok and filter_molecule(m)]
print(f"{len(survivors)} survive structure filters")
for mol in survivors[:5]:
    print(" ", mol)

## What's next

- Real dataset: replace `MY_SDF` and the `Y = ...` cell with your activity column.
- More gens: bump `NUM_GENS` to 30 and `fitness_batch_size`/`num_parents_mating` proportionally to your population size.
- Bayesian optimization instead: see `notebooks/03_bring_your_own_optimizer.ipynb`.
- Scaffold constraint: see `notebooks/04_scaffold_constrained_generation.ipynb`.
